# Langfuse Smoke Test — Compliance Pipeline

**Purpose:** Fire 10 synthetic alerts through the compliance pipeline and verify that:
1. The API responds correctly for each alert (status 200, risk_score, decision present).
2. Langfuse received and indexed the traces (verified via `/api/v1/metrics/summary`).

**Run once after deploy.** Requires the API to be running at `http://api:8000`.

In [1]:
import json
import time
import requests
import os
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "http://api:8000"

In [2]:
with open("langfuse_smoke_test_alerts.json") as f:
    alerts = json.load(f)

print(f"Loaded {len(alerts)} test alerts")

Loaded 10 test alerts


In [3]:
responses = []

for alert in alerts:
    alert_id = alert["alert_id"]
    payload = {
        "customer_id": alert["customer_id"],
        "alert_type": alert["alert_type"],
        "description": alert["description"],
    }
    r = requests.post(f"{BASE_URL}/api/v1/alerts/{alert_id}/analyze", json=payload)
    body = r.json() if r.ok else {}
    print(
        f"{alert_id} | status={r.status_code} "
        f"| decision={body.get('decision')} "
        f"| risk_score={body.get('risk_score')}"
    )
    responses.append((r.status_code, body))

TEST-001 | status=200 | decision=escalate | risk_score=7
TEST-002 | status=200 | decision=escalate | risk_score=7
TEST-003 | status=200 | decision=escalate | risk_score=8
TEST-004 | status=200 | decision=escalate | risk_score=7
TEST-005 | status=200 | decision=escalate | risk_score=7
TEST-006 | status=200 | decision=escalate | risk_score=8
TEST-007 | status=200 | decision=escalate | risk_score=7
TEST-008 | status=200 | decision=escalate | risk_score=7
TEST-009 | status=200 | decision=escalate | risk_score=7
TEST-010 | status=200 | decision=escalate | risk_score=8


In [4]:
assert all(status == 200 for status, _ in responses), "Some requests did not return 200"
assert all(body.get("risk_score") is not None for _, body in responses), "Some responses missing risk_score"
assert all(body.get("decision") is not None for _, body in responses), "Some responses missing decision"

print("\u2713 All assertions passed")

✓ All assertions passed


In [5]:
print("Waiting 60s for Langfuse ingestion...")
time.sleep(60)

Waiting 60s for Langfuse ingestion...


In [25]:
# Intento 2 — sin type
query = json.dumps({
    "view": "observations",
    "metrics": [{"measure": "latency", "aggregation": "p95"}],
    "dimensions": [],
    "filters": [{"column": "name", "operator": "=", "value": "compliance-pipeline"}],
    "fromTimestamp": "2026-04-20T00:00:00Z",
    "toTimestamp": "2026-05-20T23:59:59Z"
})
result = lf.api.metrics.metrics(query=query)
print(result)

Error: headers: {'date': 'Wed, 20 May 2026 19:50:52 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '201', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"16kww5wxk2g5l"'}, status_code: 400, body: {'message': 'Invalid request data', 'error': [{'code': 'invalid_union', 'errors': [], 'note': 'No matching discriminator', 'discriminator': 'type', 'path': ['query', 'filters', 0, 'type'], 'message': 'Invalid input'}]}

In [12]:
import langfuse
print(langfuse.__version__)

4.6.1


In [15]:
r = requests.get(f"{BASE_URL}/api/v1/metrics/summary")
summary = r.json()
print(json.dumps(summary, indent=2))

{
  "period": {
    "from": "2026-04-20",
    "to": "2026-05-20"
  },
  "latency_p95_ms": null,
  "avg_cost_per_alert_usd": null,
  "avg_tokens_per_alert": null,
  "escalation_rate": null,
  "auto_dismissed_rate": null,
  "error_rate": null,
  "risk_score": {
    "avg": null,
    "p25": null,
    "p50": null,
    "p75": null,
    "p95": null
  },
  "latency_by_agent": {
    "investigador": null,
    "risk_analyzer": null,
    "decision_agent": null
  }
}


In [7]:
assert "error" not in summary, f"Langfuse not connected: {summary.get('error')}"
print("\u2713 Langfuse connected")

✓ Langfuse connected


## Verify in Langfuse Dashboard

Check the Langfuse dashboard at https://us.cloud.langfuse.com for traces named **`compliance-pipeline`**.

Each alert should appear as one trace with:
- 3 child observations: `investigador`, `risk_analyzer`, `decision_agent`
- 4 scores: `risk_score`, `escalation_decision`, `auto_dismissed`, `confidence`

Alerts that triggered auto-escalation (risk_score ≥ 9) will have only 2 child observations
(`investigador` and `risk_analyzer`) and will be missing the `decision_agent` span.